# VWAP Mean-Reversion Strategy

**Hypothesis:** Price tends to revert to VWAP after extended deviations. Enter long when price drops >N standard deviations below VWAP; enter short (or exit) when price reverts to VWAP or exceeds +N std devs above.

**Asset:** SPY (S&P 500 ETF)  
**Timeframe:** Daily  
**Data Source:** yfinance  
**Backtest Engine:** vectorbt  

**Alpha Factory Context:**
- Source: Seed query "Anchored VWAP deviation signal intraday"
- Variant ID: `vwap-mr-spy-daily-v1`
- QC thresholds: Sharpe IS>0.8, OOS>0.6, MaxDD<25%, Calmar>0.5, 100+ trades

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nikolas-joyce/swarm-alpha-notebooks/blob/main/strategies/vwap_mean_reversion.ipynb)

## 0. Setup

In [ ]:
# ── Install deps (Colab) ─────────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q vectorbt==0.26.2 pandas-ta>=0.3.14b1 yfinance>=0.2.36
    # Clone lib/ for qc_gate and backtest_utils
    !git clone --depth 1 https://github.com/nikolas-joyce/swarm-alpha-notebooks.git /tmp/swarm-alpha 2>/dev/null || true
    sys.path.insert(0, '/tmp/swarm-alpha')
else:
    # Running locally — lib/ is already on path
    sys.path.insert(0, '..')

In [ ]:
import numpy as np
import pandas as pd
import pandas_ta as ta
import vectorbt as vbt
import yfinance as yf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from lib.qc_gate import apply_qc_gate, export_qc_result, QC_THRESHOLDS
from lib.backtest_utils import (
    split_is_oos, compute_metrics, merge_is_oos_metrics, print_comparison
)

print('All imports OK')
print(f'vectorbt: {vbt.__version__}')
print(f'QC thresholds: {QC_THRESHOLDS}')

## 1. Data Download

In [ ]:
# ── Parameters ────────────────────────────────────────────────────
TICKER = "SPY"
START_DATE = "2015-01-01"
OOS_SPLIT = 0.7           # 70% in-sample, 30% out-of-sample
INITIAL_CASH = 100_000

# Strategy parameters (will sweep later)
VWAP_LOOKBACK = 20        # Rolling VWAP window (days)
ENTRY_ZSCORE = -1.5       # Enter long when z-score below this
EXIT_ZSCORE = 0.0         # Exit when z-score reverts to this
STOP_ZSCORE = -3.0        # Stop-loss z-score level

# ── Download ──────────────────────────────────────────────────────
raw = yf.download(TICKER, start=START_DATE, auto_adjust=True)
print(f"Downloaded {len(raw)} bars: {raw.index[0].date()} → {raw.index[-1].date()}")

In [ ]:
# Flatten MultiIndex columns if present (yfinance ≥0.2.36)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

df = raw[["Open", "High", "Low", "Close", "Volume"]].copy()
df.head()

## 2. Feature Engineering — VWAP + Z-Score

In [ ]:
# ── Rolling VWAP (anchored to N-day window) ──────────────────────
# True intraday VWAP resets daily; for daily bars we use a rolling
# volume-weighted average price as a proxy.

typical_price = (df["High"] + df["Low"] + df["Close"]) / 3
tp_vol = typical_price * df["Volume"]

df["vwap"] = tp_vol.rolling(VWAP_LOOKBACK).sum() / df["Volume"].rolling(VWAP_LOOKBACK).sum()

# ── VWAP deviation z-score ────────────────────────────────────────
deviation = df["Close"] - df["vwap"]
df["vwap_zscore"] = (
    deviation / deviation.rolling(VWAP_LOOKBACK).std()
)

# ── Additional indicators (for context, not signals) ─────────────
df["rsi"] = df.ta.rsi(length=14)
df["atr"] = df.ta.atr(length=14)
df["sma_200"] = df.ta.sma(length=200)

df.dropna(inplace=True)
print(f"Feature matrix: {df.shape[0]} bars × {df.shape[1]} columns")
df[["Close", "vwap", "vwap_zscore"]].tail(10)

## 3. Signal Generation

In [ ]:
# ── Long-only mean-reversion signals ─────────────────────────────
# Entry: z-score crosses below ENTRY_ZSCORE (oversold vs VWAP)
# Exit:  z-score crosses above EXIT_ZSCORE (reverted to mean)
# Stop:  z-score drops below STOP_ZSCORE (trend, not mean-reversion)

entries = (
    (df["vwap_zscore"] < ENTRY_ZSCORE) &
    (df["vwap_zscore"].shift(1) >= ENTRY_ZSCORE)  # cross, not just below
)

exits = (
    (df["vwap_zscore"] > EXIT_ZSCORE) |
    (df["vwap_zscore"] < STOP_ZSCORE)
)

print(f"Entry signals: {entries.sum()}")
print(f"Exit signals:  {exits.sum()}")

## 4. IS/OOS Split + Backtest

In [ ]:
# ── Split ─────────────────────────────────────────────────────────
split_idx = int(len(df) * OOS_SPLIT)
split_date = df.index[split_idx]
print(f"IS: {df.index[0].date()} → {split_date.date()} ({split_idx} bars)")
print(f"OOS: {split_date.date()} → {df.index[-1].date()} ({len(df) - split_idx} bars)")

# IS
price_is = df["Close"].iloc[:split_idx]
entries_is = entries.iloc[:split_idx]
exits_is = exits.iloc[:split_idx]

# OOS
price_oos = df["Close"].iloc[split_idx:]
entries_oos = entries.iloc[split_idx:]
exits_oos = exits.iloc[split_idx:]

In [ ]:
# ── Run backtests ─────────────────────────────────────────────────
pf_is = vbt.Portfolio.from_signals(
    price_is, entries_is, exits_is,
    init_cash=INITIAL_CASH,
    freq="1D",
)

pf_oos = vbt.Portfolio.from_signals(
    price_oos, entries_oos, exits_oos,
    init_cash=INITIAL_CASH,
    freq="1D",
)

# ── Extract metrics ───────────────────────────────────────────────
is_metrics = compute_metrics(pf_is, label="In-Sample")
oos_metrics = compute_metrics(pf_oos, label="Out-of-Sample")

print_comparison(is_metrics, oos_metrics)

## 5. QC Gate Evaluation

In [ ]:
# ── Merge into QC-gate format and evaluate ───────────────────────
qc_input = merge_is_oos_metrics(is_metrics, oos_metrics)

print("\nQC Gate Input:")
for k, v in qc_input.items():
    print(f"  {k}: {v}")

passed, failures = apply_qc_gate(qc_input)

print(f"\n{'='*40}")
if passed:
    print("✓ QC GATE: PASSED")
    print("  → Strategy qualifies for vault (tested/)")
else:
    print("✗ QC GATE: FAILED")
    for f in failures:
        print(f"  ✗ {f}")
    print("  → Strategy goes to graveyard/")
print(f"{'='*40}")

## 6. Visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# Price + VWAP
axes[0].plot(df.index, df["Close"], label="Close", linewidth=0.8)
axes[0].plot(df.index, df["vwap"], label=f"VWAP({VWAP_LOOKBACK})", linewidth=0.8, alpha=0.8)
axes[0].axvline(split_date, color="red", linestyle="--", label="IS/OOS Split", alpha=0.7)
axes[0].set_title(f"{TICKER} — VWAP Mean-Reversion")
axes[0].legend(loc="upper left")
axes[0].set_ylabel("Price")

# Z-score
axes[1].plot(df.index, df["vwap_zscore"], linewidth=0.7, color="purple")
axes[1].axhline(ENTRY_ZSCORE, color="green", linestyle="--", alpha=0.6, label=f"Entry ({ENTRY_ZSCORE})")
axes[1].axhline(EXIT_ZSCORE, color="gray", linestyle="--", alpha=0.6, label=f"Exit ({EXIT_ZSCORE})")
axes[1].axhline(STOP_ZSCORE, color="red", linestyle="--", alpha=0.6, label=f"Stop ({STOP_ZSCORE})")
axes[1].axvline(split_date, color="red", linestyle="--", alpha=0.7)
axes[1].set_ylabel("VWAP Z-Score")
axes[1].legend(loc="lower left")
axes[1].set_ylim(-4, 4)

# Equity curve (combined, marking split)
# Rebuild full backtest for visualization
pf_full = vbt.Portfolio.from_signals(
    df["Close"], entries, exits, init_cash=INITIAL_CASH, freq="1D"
)
equity = pf_full.value()
axes[2].plot(equity.index, equity.values, linewidth=0.8, color="darkblue")
axes[2].axvline(split_date, color="red", linestyle="--", alpha=0.7)
axes[2].axhline(INITIAL_CASH, color="gray", linestyle=":", alpha=0.5)
axes[2].set_ylabel("Portfolio Value ($)")
axes[2].set_xlabel("Date")

plt.tight_layout()
plt.show()

## 7. Parameter Sweep (Optional)

Sweep entry z-score and VWAP lookback to find robust parameter regions.

In [ ]:
# ── Sweep entry z-score × VWAP lookback on IS data only ──────────
lookbacks = [10, 15, 20, 30, 40]
z_entries = [-1.0, -1.25, -1.5, -1.75, -2.0, -2.5]

sweep_results = []

for lb in lookbacks:
    # Recompute VWAP + z-score for this lookback
    tp = (df["High"] + df["Low"] + df["Close"]) / 3
    tv = tp * df["Volume"]
    vwap_lb = tv.rolling(lb).sum() / df["Volume"].rolling(lb).sum()
    dev = df["Close"] - vwap_lb
    zscore_lb = dev / dev.rolling(lb).std()

    for z_entry in z_entries:
        ent = (
            (zscore_lb < z_entry) &
            (zscore_lb.shift(1) >= z_entry)
        )
        ext = zscore_lb > 0  # exit at mean

        # IS only
        ent_is = ent.iloc[:split_idx].dropna()
        ext_is = ext.iloc[:split_idx].dropna()
        p_is = df["Close"].iloc[:split_idx]

        # Align indices
        common = p_is.index.intersection(ent_is.index).intersection(ext_is.index)
        if len(common) < 50:
            continue

        try:
            pf = vbt.Portfolio.from_signals(
                p_is.loc[common], ent_is.loc[common], ext_is.loc[common],
                init_cash=INITIAL_CASH, freq="1D"
            )
            stats = pf.stats()
            sweep_results.append({
                "lookback": lb,
                "z_entry": z_entry,
                "sharpe": float(stats.get("Sharpe Ratio", 0)),
                "max_dd": abs(float(stats.get("Max Drawdown [%]", 100))) / 100,
                "trades": int(stats.get("Total Trades", 0)),
                "total_return": float(stats.get("Total Return [%]", 0)) / 100,
            })
        except Exception:
            pass

sweep_df = pd.DataFrame(sweep_results)
if len(sweep_df) > 0:
    pivot = sweep_df.pivot_table(
        values="sharpe", index="z_entry", columns="lookback"
    )
    print("\nSharpe Ratio Heatmap (IS only):")
    print(pivot.round(3).to_string())
    print(f"\nBest combo: {sweep_df.loc[sweep_df['sharpe'].idxmax()].to_dict()}")
else:
    print("No valid sweep results — check data range or parameters")

## 8. Export QC Result for Alpha Factory Station 5

In [ ]:
# ── Export ─────────────────────────────────────────────────────────
# This JSON is compatible with obsidian_writer.py
# Copy to 12-Alpha-Factory/data/results/ for Station 5 pickup

result_path = export_qc_result(
    strategy_name="VWAP Mean-Reversion (SPY Daily)",
    variant_id="vwap-mr-spy-daily-v1",
    results=qc_input,
    passed=passed,
    failures=failures,
    output_dir="../results" if not IN_COLAB else "/content/results",
)

print(f"\nResult exported to: {result_path}")
print("\nNext steps:")
if IN_COLAB:
    print("  1. Download the JSON from /content/results/")
    print("  2. Copy to 12-Alpha-Factory/data/results/")
    print("  3. Run: python -m pipeline.obsidian_writer")
else:
    print("  1. Copy results/*.json to 12-Alpha-Factory/data/results/")
    print("  2. Run: python -m pipeline.obsidian_writer")

In [ ]:
# ── Download helper (Colab only) ─────────────────────────────────
if IN_COLAB:
    from google.colab import files
    files.download(str(result_path))

---

## Notes

**Known limitations of this variant:**
- Daily VWAP is a proxy — true VWAP resets intraday and requires tick data
- Long-only; a short leg on +z-score excursions could improve risk-adjusted returns
- No regime filter — mean-reversion fails in trending markets (add SMA200 or VIX filter)
- Fixed z-score thresholds — could adapt based on ATR or realized vol

**Potential improvements (next variants):**
1. Add VIX regime filter (only trade when VIX < 25)
2. Add trend filter (only trade when Close > SMA200)
3. Adaptive z-score threshold based on rolling ATR percentile
4. Multi-asset version (QQQ, IWM, TLT) for diversification